## PACOTE ##

In [ ]:
import io, re, html, time, inspect, base64, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import confusion_matrix, average_precision_score, matthews_corrcoef, log_loss

warnings.filterwarnings("ignore")

## CÓDIGO ##

In [ ]:

AZUL = "#2563eb"
AZUL_CLARO = "#60a5fa"
AZUL_BORDA = "#1e3a8a"
AMARELO = "#facc15"
PRETO = "#111827"

def sanitizar_nome(x):
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(x))


def formatar_count(x):
    return f"{int(x):,}".replace(",", ".")

def formatar_float_html(x, casas=6):
    if pd.isna(x):
        return "-"
    try:
        return f"{float(x):.{casas}f}".replace(".", ",")
    except Exception:
        return str(x)

def formatar_float_latex(x, casas=6):
    if pd.isna(x):
        return "-"
    try:
        return f"{float(x):.{casas}f}"
    except Exception:
        return str(x)

def formatar_tempo(segundos):
    segundos = int(segundos)
    h = segundos // 3600
    m = (segundos % 3600) // 60
    s = segundos % 60
    if h:
        return f"{h}h {m}min {s}s"
    if m:
        return f"{m}min {s}s"
    return f"{s}s"

def detectar_target(df):
    if "status_fraude" in df.columns:
        return "status_fraude"
    if "Class" in df.columns:
        return "Class"
    raise ValueError("Não encontrei a coluna target: 'status_fraude' ou 'Class'.")


def valor_linha(linha, col, padrao=None):
    if col in linha.index and not pd.isna(linha[col]):
        return linha[col]
    return padrao


def scaler_bool(v):
    if pd.isna(v):
        return True
    s = str(v).strip().lower()
    if s in ["standardscaler", "standard_scaler", "sim", "true", "1", "yes"]:
        return True
    if s in ["none", "nao", "não", "false", "0", "no"]:
        return False
    return True


def fig_to_b64(fig):
    bio = io.BytesIO()
    fig.savefig(bio, format="png", dpi=150, bbox_inches="tight")
    bio.seek(0)
    return base64.b64encode(bio.read()).decode("utf-8")


def salvar_fig(fig, path_sem_ext):
    p = Path(path_sem_ext)
    fig.savefig(p.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(p.with_suffix(".png"), dpi=300, bbox_inches="tight")


def criar_tsne_1d(perplexity, random_state=42, init="pca", max_iter=250,
                  learning_rate="auto", n_jobs=-1, verbose=0,
                  method="barnes_hut", angle=0.5):
    params = inspect.signature(TSNE).parameters
    kwargs = dict(
        n_components=1,
        perplexity=perplexity,
        random_state=random_state,
        init=init,
        learning_rate=learning_rate,
        method=method,
        angle=angle,
        verbose=verbose,
    )
    if "max_iter" in params:
        kwargs["max_iter"] = max_iter
    else:
        kwargs["n_iter"] = max_iter
    if "n_jobs" in params:
        kwargs["n_jobs"] = n_jobs
    return TSNE(**kwargs)

def gerar_tabela_html(df, table_id):
    if df is None or df.empty:
        return "<p>Nenhum dado disponível.</p>"
    s = f'<table id="{html.escape(str(table_id))}" class="data-table"><thead><tr>'
    for c in df.columns:
        s += f"<th>{html.escape(str(c))}</th>"
    s += "</tr></thead><tbody>"
    for _, row in df.iterrows():
        s += "<tr>" + "".join(f"<td>{html.escape(str(v))}</td>" for v in row) + "</tr>"
    s += "</tbody></table>"
    return s

def secao_tabela(titulo, tabela_html, table_id, nome_csv):
    return f'''
    <section class="plot-card">
      <div class="section-header">
        <h2>{html.escape(str(titulo))}</h2>
        <button class="download-btn" onclick="baixarTabelaCSV('{html.escape(str(table_id))}', '{html.escape(str(nome_csv))}')">Baixar CSV</button>
      </div>
      <div class="table-wrapper">{tabela_html}</div>
    </section>
    '''

def secao_imagem(titulo, b64, nome_download, alt):
    return f'''
    <section class="plot-card">
      <div class="section-header">
        <h2>{html.escape(str(titulo))}</h2>
        <a class="download-btn" href="data:image/png;base64,{b64}" download="{html.escape(str(nome_download))}">Baixar PNG</a>
      </div>
      <img class="plot-img" src="data:image/png;base64,{b64}" alt="{html.escape(str(alt))}">
    </section>
    '''

def preparar_latex(df):
    out = df.copy()
    for c in out.columns:
        if pd.api.types.is_float_dtype(out[c]):
            out[c] = out[c].apply(lambda x: formatar_float_latex(x, 6))
        elif pd.api.types.is_integer_dtype(out[c]):
            out[c] = out[c].apply(lambda x: int(x) if not pd.isna(x) else x)
    return out

def salvar_tabela_latex(df, caminho, caption, label, longtable=False):
    caminho = Path(caminho)
    if df is None or df.empty:
        caminho.write_text("% Tabela vazia.\n", encoding="utf-8")
        return
    tex = preparar_latex(df).to_latex(index=False, escape=True, longtable=longtable, caption=caption, label=label)
    caminho.write_text(tex, encoding="utf-8")


def exportar_latex(pasta, tabela_metricas, tabela_matrizes):
    pasta = Path(pasta)
    pasta.mkdir(parents=True, exist_ok=True)
    salvar_tabela_latex(tabela_metricas, pasta / "tabela_metricas_ranks.tex", "Métricas dos melhores rankings do experimento t-SNE 1D.", "tab:metricas-ranks-tsne-1d", False)
    salvar_tabela_latex(tabela_matrizes, pasta / "tabela_matrizes_confusao_ranks.tex", "Matrizes de confusão dos melhores rankings do experimento t-SNE 1D.", "tab:matrizes-ranks-tsne-1d", True)
    comandos = r'''

% \usepackage{graphicx}
% \usepackage{float}
% \usepackage{booktabs}
% \usepackage{longtable}
% \usepackage{pdflscape}

\begin{table}[H]
\centering
\caption{Métricas dos melhores rankings do experimento t-SNE 1D.}
\label{tab:metricas-ranks-tsne-1d-main}
\input{1x1_tsne_visu_scores/tabela_metricas_ranks.tex}
\end{table}

\begin{landscape}
\small
\input{1x1_tsne_visu_scores/tabela_matrizes_confusao_ranks.tex}
\end{landscape}

\begin{figure}[H]
\centering
\includegraphics[width=\textwidth]{1x1_tsne_visu_scores/rank_1_tsne1_perplexity_EXEMPLO_representacao_1d.pdf}
\caption{Representação 1D do t-SNE no Rank 1.}
\label{fig:rank1-tsne1-representacao}
\end{figure}
'''
    (pasta / "comandos_latex_exemplo.tex").write_text(comandos, encoding="utf-8")

def matriz_confusao(y, prob, threshold):
    pred = (prob >= threshold).astype(int)
    return confusion_matrix(y, pred, labels=[0, 1])

def valores_matriz(cm):
    tn, fp, fn, tp = cm.ravel()
    total_fraudes = fn + tp
    total_nao = tn + fp
    fn_pct = fn / total_fraudes * 100 if total_fraudes else 0
    tp_pct = tp / total_fraudes * 100 if total_fraudes else 0
    tn_pct = tn / total_nao * 100 if total_nao else 0
    fp_pct = fp / total_nao * 100 if total_nao else 0
    return {
        "fn": {"pct": fn_pct, "count": int(fn), "qualidade": 100 - fn_pct},
        "tp": {"pct": tp_pct, "count": int(tp), "qualidade": tp_pct},
        "tn": {"pct": tn_pct, "count": int(tn), "qualidade": tn_pct},
        "fp": {"pct": fp_pct, "count": int(fp), "qualidade": 100 - fp_pct},
        "raw": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }


def valores_matriz_ideal(y):
    y = np.asarray(y).astype(int)
    n_fraude = int(np.sum(y == 1))
    n_nao = int(np.sum(y == 0))
    return {
        "fn": {"pct": 0.0, "count": 0, "qualidade": 100.0},
        "tp": {"pct": 100.0, "count": n_fraude, "qualidade": 100.0},
        "tn": {"pct": 100.0, "count": n_nao, "qualidade": 100.0},
        "fp": {"pct": 0.0, "count": 0, "qualidade": 100.0},
        "raw": {"tn": n_nao, "fp": 0, "fn": 0, "tp": n_fraude},
    }


def classe_qualidade(q):
    if q >= 95: return "cell q95"
    if q >= 85: return "cell q85"
    if q >= 70: return "cell q70"
    if q >= 50: return "cell q50"
    if q >= 30: return "cell q30"
    return "cell q10"


def html_matriz(titulo, v, matriz_ideal=False, b64=None, nome_img=None):
    if matriz_ideal:
        desc_fn, desc_tp, desc_tn, desc_fp = "Erro ideal: nenhuma fraude perdida", "Acerto ideal: fraudes detectadas", "Acerto ideal: não fraudes corretas", "Erro ideal: nenhum falso alerta"
    else:
        desc_fn, desc_tp, desc_tn, desc_fp = "Erro: fraude perdida", "Acerto: fraude detectada", "Acerto: não fraude", "Erro: falso alerta"
    botao = ""
    if b64 is not None:
        botao = f'<div class="section-actions-only"><a class="download-btn" href="data:image/png;base64,{b64}" download="{html.escape(str(nome_img))}">Baixar PNG</a></div>'
    return f'''
    <section class="matrix-card">
      <h2>{html.escape(str(titulo))}</h2>
      {botao}
      <div class="matrix-area">
        <div class="matrix-wrapper">
          <div class="corner"></div><div class="x-label">Pred Não Fraude</div><div class="x-label">Pred Fraude</div>
          <div class="y-label">Real Fraude</div>
          <div class="{classe_qualidade(v['fn']['qualidade'])}"><div class="pct">{v['fn']['pct']:.2f}%</div><div class="count">({formatar_count(v['fn']['count'])})</div><div class="cell-desc">{desc_fn}</div></div>
          <div class="{classe_qualidade(v['tp']['qualidade'])}"><div class="pct">{v['tp']['pct']:.2f}%</div><div class="count">({formatar_count(v['tp']['count'])})</div><div class="cell-desc">{desc_tp}</div></div>
          <div class="y-label">Real Não Fraude</div>
          <div class="{classe_qualidade(v['tn']['qualidade'])}"><div class="pct">{v['tn']['pct']:.2f}%</div><div class="count">({formatar_count(v['tn']['count'])})</div><div class="cell-desc">{desc_tn}</div></div>
          <div class="{classe_qualidade(v['fp']['qualidade'])}"><div class="pct">{v['fp']['pct']:.2f}%</div><div class="count">({formatar_count(v['fp']['count'])})</div><div class="cell-desc">{desc_fp}</div></div>
        </div>
        <div class="legend"><div class="legend-title">Qualidade</div><div class="colorbar"></div><div class="legend-label-top">Melhor</div><div class="legend-label-bottom">Pior</div></div>
      </div>
    </section>
    '''


def fig_matriz(titulo, v):
    pct = np.array([[v["fn"]["pct"], v["tp"]["pct"]], [v["tn"]["pct"], v["fp"]["pct"]]])
    cnt = np.array([[v["fn"]["count"], v["tp"]["count"]], [v["tn"]["count"], v["fp"]["count"]]])
    textos = [["Fraude perdida", "Fraude detectada"], ["Não fraude correta", "Falso alerta"]]
    fig, ax = plt.subplots(figsize=(8.5, 6.2))
    im = ax.imshow(pct, vmin=0, vmax=100, cmap="Blues")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Percentual por classe real (%)", fontweight="bold")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred Não Fraude", "Pred Fraude"], fontweight="bold")
    ax.set_yticklabels(["Real Fraude", "Real Não Fraude"], fontweight="bold")
    ax.set_title(titulo, fontsize=14, fontweight="bold", pad=14)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{pct[i,j]:.2f}%\n({formatar_count(cnt[i,j])})\n{textos[i][j]}", ha="center", va="center", color="black", fontweight="bold", fontsize=10)
    plt.tight_layout()
    return fig

def fig_boxplot(df_plot, target_name, titulo):
    nf = df_plot.loc[df_plot[target_name] == 0, "TSNE_1"].dropna()
    fr = df_plot.loc[df_plot[target_name] == 1, "TSNE_1"].dropna()
    fig, ax = plt.subplots(figsize=(10, 6))
    box = ax.boxplot([nf, fr], labels=[f"Não Fraude ({formatar_count(len(nf))})", f"Fraude ({formatar_count(len(fr))})"], patch_artist=True, showfliers=True)
    for patch, cor in zip(box["boxes"], [AZUL, AMARELO]):
        patch.set_facecolor(cor); patch.set_alpha(0.72); patch.set_linewidth(2)
    for median in box["medians"]:
        median.set_color(PRETO); median.set_linewidth(2.3)
    for flier in box["fliers"]:
        flier.set_marker("o"); flier.set_markerfacecolor("#64748b"); flier.set_markeredgecolor("#64748b"); flier.set_alpha(0.20); flier.set_markersize(2.5)
    ax.set_title(f"Boxplot por Classe - {titulo}", fontsize=14, fontweight="bold")
    ax.set_ylabel("TSNE_1", fontsize=14, fontweight="bold")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    return fig

def fig_representacao(df_plot, target_name, titulo):
    nf = df_plot.loc[df_plot[target_name] == 0, "TSNE_1"].dropna()
    fr = df_plot.loc[df_plot[target_name] == 1, "TSNE_1"].dropna()
    rng = np.random.default_rng(42)
    y_nf = rng.normal(0, 0.025, len(nf))
    y_fr = rng.normal(1, 0.035, len(fr))
    fig, ax = plt.subplots(figsize=(11, 4.8))
    ax.scatter(nf, y_nf, s=6, alpha=0.06, color=AZUL, rasterized=True)
    ax.scatter(fr, y_fr, s=32, alpha=0.88, color=AMARELO, edgecolors=PRETO, linewidths=0.25, rasterized=True)
    ax.set_title(f"Representação 1D com 100% dos Dados - {titulo}", fontsize=14, fontweight="bold")
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Não Fraude", "Fraude"], fontsize=12, fontweight="bold")
    ax.set_xlabel("TSNE_1", fontsize=14, fontweight="bold"); ax.set_ylabel("Classe Real", fontsize=13, fontweight="bold")
    ax.grid(alpha=0.22)
    h1 = plt.Line2D([0], [0], marker="o", linestyle="", markersize=8, markerfacecolor=AZUL, markeredgecolor=AZUL_BORDA, label=f"Não Fraude ({formatar_count(len(nf))})")
    h2 = plt.Line2D([0], [0], marker="o", linestyle="", markersize=8, markerfacecolor=AMARELO, markeredgecolor=PRETO, label=f"Fraude ({formatar_count(len(fr))})")
    ax.legend(handles=[h1, h2], title="Classe Real", loc="best", frameon=True)
    plt.tight_layout()
    return fig

def fig_spearman(df_plot, target_name, titulo):
    corr = df_plot[["TSNE_1", target_name]].rename(columns={target_name: "Fraude"}).corr(method="spearman")
    labels = corr.columns.tolist()
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Correlação de Spearman", fontsize=11, fontweight="bold")
    ax.set_xticks(np.arange(len(labels))); ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, fontsize=12, fontweight="bold", rotation=35, ha="right")
    ax.set_yticklabels(labels, fontsize=12, fontweight="bold")
    ax.set_title(f"Correlação de Spearman - {titulo}", fontsize=13, fontweight="bold")
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{corr.values[i, j]:.3f}", ha="center", va="center", color="black", fontsize=13, fontweight="bold")
    plt.tight_layout()
    return fig

def pontos_corte_1d(x_grid, prob_grid, threshold):
    diff = prob_grid - threshold
    idx = np.where(np.sign(diff[:-1]) != np.sign(diff[1:]))[0]
    pts = []
    for i in idx:
        x1, x2 = x_grid[i], x_grid[i + 1]
        y1, y2 = diff[i], diff[i + 1]
        pts.append(x1 if y2 == y1 else x1 - y1 * (x2 - x1) / (y2 - y1))
    return pts

def fig_responsabilidades(df_plot, target_name, scaler, gmm, cluster_fraude, prob, threshold, titulo):
    y = df_plot[target_name].astype(int).to_numpy()
    x = df_plot["TSNE_1"].to_numpy()
    mask_nf, mask_fr = y == 0, y == 1
    xmin, xmax = np.min(x), np.max(x)
    margem = 0.05 * (xmax - xmin) if xmax != xmin else 1.0
    x_grid = np.linspace(xmin - margem, xmax + margem, 1200)
    x_grid_df = pd.DataFrame({"TSNE_1": x_grid})
    x_scaled = scaler.transform(x_grid_df)
    prob_grid = gmm.predict_proba(x_scaled)[:, cluster_fraude]
    cortes = pontos_corte_1d(x_grid, prob_grid, threshold)
    fig, ax = plt.subplots(figsize=(11, 6.2))
    ax.scatter(x[mask_nf], prob[mask_nf], s=7, alpha=0.15, color=AZUL_CLARO, rasterized=True)
    ax.scatter(x[mask_fr], prob[mask_fr], s=34, alpha=0.90, color=AMARELO, edgecolors=PRETO, linewidths=0.25, rasterized=True)
    ax.plot(x_grid, prob_grid, color="#dc2626", linewidth=2.3, label="Responsabilidade estimada pela GMM")
    ax.axhline(threshold, color=PRETO, linestyle="--", linewidth=2.0, label="Ponto de corte")
    for p in cortes:
        ax.axvline(p, color=PRETO, linestyle=":", linewidth=1.8, alpha=0.95)
    ax.set_title(f"{titulo} | Corte = {threshold:.6f}", fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("TSNE_1", fontsize=14, fontweight="bold"); ax.set_ylabel("Responsabilidade GMM para fraude", fontsize=13, fontweight="bold")
    ax.set_ylim(-0.03, 1.03); ax.grid(alpha=0.22)
    h_nf = plt.Line2D([0], [0], marker="o", linestyle="", markersize=8, markerfacecolor=AZUL_CLARO, markeredgecolor=AZUL_BORDA, label=f"Não Fraude real ({formatar_count(mask_nf.sum())})")
    h_fr = plt.Line2D([0], [0], marker="o", linestyle="", markersize=8, markerfacecolor=AMARELO, markeredgecolor=PRETO, label=f"Fraude real ({formatar_count(mask_fr.sum())})")
    h_v = plt.Line2D([0], [0], linestyle=":", linewidth=1.8, color=PRETO, label="Ponto(s) onde responsabilidade = corte")
    handles, _ = ax.get_legend_handles_labels()
    ax.legend(handles=[h_nf, h_fr, handles[0], handles[1], h_v], title="Legenda", loc="best", frameon=True)
    plt.tight_layout()
    return fig

def processar_rank(rank, df, scores, target_name, features_tsne, pasta_latex, verbose_tsne=0):
    if "Posicao_Rank" not in scores.columns:
        scores = scores.sort_values("Score_Final", ascending=False).reset_index(drop=True)
        scores["Posicao_Rank"] = np.arange(1, len(scores) + 1)
    if rank not in scores["Posicao_Rank"].values:
        raise ValueError(f"Rank {rank} não encontrado.")

    linha = scores.loc[scores["Posicao_Rank"] == rank].iloc[0]

    perplexity = float(valor_linha(linha, "Perplexity"))
    max_iter = int(float(valor_linha(linha, "Max_Iter", 250)))
    init = str(valor_linha(linha, "Init", "pca"))
    learning_rate = valor_linha(linha, "Learning_Rate", "auto")
    if str(learning_rate).replace('.', '', 1).isdigit():
        learning_rate = float(learning_rate)
    random_state_tsne = int(float(valor_linha(linha, "Random_State_TSNE", 42)))
    method_tsne = str(valor_linha(linha, "Method_TSNE", "barnes_hut"))
    angle_tsne = float(valor_linha(linha, "Angle_TSNE", 0.5))
    n_jobs_tsne = int(float(valor_linha(linha, "N_Jobs_TSNE", -1)))
    escalar_antes = scaler_bool(valor_linha(linha, "Scaler_Antes_TSNE", "StandardScaler"))
    escalar_gmm = scaler_bool(valor_linha(linha, "Scaler_TSNE_Para_GMM", "StandardScaler"))

    gmm_n = int(float(valor_linha(linha, "GMM_N_Components", 2)))
    gmm_cov = str(valor_linha(linha, "GMM_Covariance_Type", "full"))
    gmm_n_init = int(float(valor_linha(linha, "GMM_N_Init", 3)))
    gmm_rs = int(float(valor_linha(linha, "GMM_Random_State", 42)))
    gmm_reg = float(valor_linha(linha, "GMM_Reg_Covar", 1e-6))

    melhor_corte = float(valor_linha(linha, "Melhor_Ponto_Corte", 0.5))
    corte_medio = float(valor_linha(linha, "Ponto_Corte_Medio", 0.5))

    print("=" * 80)
    print(f"Recriando t-SNE 1D do Rank {rank} | perplexity={perplexity:g}")
    print("=" * 80)
    t0 = time.time()

    dados = df[features_tsne + [target_name]].dropna().copy()
    X = dados[features_tsne].copy()
    y = dados[target_name].astype(int).to_numpy()

    if escalar_antes:
        X_in = StandardScaler().fit_transform(X)
    else:
        X_in = X.to_numpy()

    tsne = criar_tsne_1d(
        perplexity=perplexity,
        random_state=random_state_tsne,
        init=init,
        max_iter=max_iter,
        learning_rate=learning_rate,
        n_jobs=n_jobs_tsne,
        verbose=verbose_tsne,
        method=method_tsne,
        angle=angle_tsne,
    )
    X_tsne = np.asarray(tsne.fit_transform(X_in))
    if X_tsne.shape[1] != 1:
        raise ValueError(f"t-SNE deveria gerar 1 componente; gerou {X_tsne.shape}.")

    temp = pd.DataFrame({"TSNE_1": X_tsne[:, 0], target_name: y})

    scaler_gmm = StandardScaler()
    X_gmm = scaler_gmm.fit_transform(temp[["TSNE_1"]]) if escalar_gmm else scaler_gmm.fit_transform(temp[["TSNE_1"]])

    gmm = GaussianMixture(n_components=gmm_n, covariance_type=gmm_cov, random_state=gmm_rs, n_init=gmm_n_init, reg_covar=gmm_reg)
    gmm.fit(X_gmm)
    clusters = gmm.predict(X_gmm)
    ct = pd.crosstab(clusters, y)
    if 1 not in ct.columns:
        raise ValueError("A classe fraude, valor 1, não foi encontrada no target.")
    cluster_fraude = int(ct[1].idxmax())
    prob = np.clip(gmm.predict_proba(X_gmm)[:, cluster_fraude], 1e-15, 1 - 1e-15)

    auc_pr_calc = average_precision_score(y, prob)
    mcc_calc = matthews_corrcoef(y, (prob >= melhor_corte).astype(int))
    logloss_calc = log_loss(y, prob, labels=[0, 1])
    logloss_norm_calc = 1 / (1 + logloss_calc)

    cm_melhor = matriz_confusao(y, prob, melhor_corte)
    cm_medio = matriz_confusao(y, prob, corte_medio)
    v_melhor = valores_matriz(cm_melhor)
    v_medio = valores_matriz(cm_medio)
    v_ideal = valores_matriz_ideal(y)

    tempo = formatar_tempo(time.time() - t0)
    prefixo = f"rank_{rank}_tsne1_perplexity_{sanitizar_nome(perplexity)}"
    titulo_base = f"t-SNE 1D | Perplexity = {perplexity:g}"

    figs = {
        "boxplot": fig_boxplot(temp, target_name, titulo_base),
        "representacao_1d": fig_representacao(temp, target_name, titulo_base),
        "spearman": fig_spearman(temp, target_name, f"TSNE_1 e Fraude | Perplexity = {perplexity:g}"),
        "responsabilidades_melhor_corte": fig_responsabilidades(temp, target_name, scaler_gmm, gmm, cluster_fraude, prob, melhor_corte, f"Responsabilidades GMM - Melhor Ponto de Corte | {titulo_base}"),
        "responsabilidades_corte_medio": fig_responsabilidades(temp, target_name, scaler_gmm, gmm, cluster_fraude, prob, corte_medio, f"Responsabilidades GMM - Ponto de Corte Médio | {titulo_base}"),
        "matriz_melhor_corte": fig_matriz(f"Rank {rank} - t-SNE 1D - Perplexity {perplexity:g} - Melhor Ponto de Corte", v_melhor),
        "matriz_corte_medio": fig_matriz(f"Rank {rank} - t-SNE 1D - Perplexity {perplexity:g} - Ponto de Corte 0.5", v_medio),
        "matriz_ideal": fig_matriz(f"Rank {rank} - t-SNE 1D - Perplexity {perplexity:g} - Matriz Ideal", v_ideal),
    }

    imgs = {}
    for nome, fig in figs.items():
        arq = f"{prefixo}_{nome}"
        salvar_fig(fig, Path(pasta_latex) / arq)
        imgs[nome] = fig_to_b64(fig)
        plt.close(fig)

    tabela_metricas = pd.DataFrame([{
        "Rank": rank,
        "Origem": "t-SNE 1D",
        "Feature": "TSNE_1",
        "Perplexity": perplexity,
        "N_Components_TSNE": 1,
        "Max_Iter": max_iter,
        "N_Iter_Real": valor_linha(linha, "N_Iter_Real", np.nan),
        "Init": init,
        "Learning_Rate": learning_rate,
        "Random_State_TSNE": random_state_tsne,
        "Method_TSNE": method_tsne,
        "Angle_TSNE": angle_tsne,
        "N_Jobs_TSNE": n_jobs_tsne,
        "Scaler_Antes_TSNE": "StandardScaler" if escalar_antes else "None",
        "Scaler_TSNE_Para_GMM": "StandardScaler" if escalar_gmm else "None",
        "Melhor_Ponto_Corte": melhor_corte,
        "Ponto_Corte_Medio": corte_medio,
        "AUC_PR_CSV": float(valor_linha(linha, "AUC_PR", np.nan)),
        "AUC_PR_Recalculado": float(auc_pr_calc),
        "MCC_CSV": float(valor_linha(linha, "MCC", np.nan)),
        "MCC_Recalculado": float(mcc_calc),
        "Log_Loss_Norm_CSV": float(valor_linha(linha, "Log_Loss_Norm", np.nan)),
        "Log_Loss_Norm_Recalculado": float(logloss_norm_calc),
        "Score_Final": float(valor_linha(linha, "Score_Final", np.nan)),
        "Diferenca_Neg_Log_Veross": float(valor_linha(linha, "Diferenca_Neg_Log_Veross", np.nan)),
        "GMM_N_Components": gmm_n,
        "GMM_Covariance_Type": gmm_cov,
        "GMM_Random_State": gmm_rs,
        "GMM_N_Init": gmm_n_init,
        "GMM_Reg_Covar": gmm_reg,
        "Cluster_Fraude": cluster_fraude,
        "Tempo_Recriacao_TSNE_e_Relatorio": tempo,
    }])

    tabela_matrizes = pd.DataFrame([
        {"Rank": rank, "Origem": "t-SNE 1D", "Perplexity": perplexity, "Cenario": "Melhor ponto de corte", "Threshold": melhor_corte, "TN": v_melhor["raw"]["tn"], "FP": v_melhor["raw"]["fp"], "FN": v_melhor["raw"]["fn"], "TP": v_melhor["raw"]["tp"], "FN_Pct_Real_Fraude": v_melhor["fn"]["pct"], "TP_Pct_Real_Fraude": v_melhor["tp"]["pct"], "TN_Pct_Real_Nao_Fraude": v_melhor["tn"]["pct"], "FP_Pct_Real_Nao_Fraude": v_melhor["fp"]["pct"]},
        {"Rank": rank, "Origem": "t-SNE 1D", "Perplexity": perplexity, "Cenario": "Ponto de corte médio", "Threshold": corte_medio, "TN": v_medio["raw"]["tn"], "FP": v_medio["raw"]["fp"], "FN": v_medio["raw"]["fn"], "TP": v_medio["raw"]["tp"], "FN_Pct_Real_Fraude": v_medio["fn"]["pct"], "TP_Pct_Real_Fraude": v_medio["tp"]["pct"], "TN_Pct_Real_Nao_Fraude": v_medio["tn"]["pct"], "FP_Pct_Real_Nao_Fraude": v_medio["fp"]["pct"]},
        {"Rank": rank, "Origem": "t-SNE 1D", "Perplexity": perplexity, "Cenario": "Ideal", "Threshold": np.nan, "TN": v_ideal["raw"]["tn"], "FP": v_ideal["raw"]["fp"], "FN": v_ideal["raw"]["fn"], "TP": v_ideal["raw"]["tp"], "FN_Pct_Real_Fraude": v_ideal["fn"]["pct"], "TP_Pct_Real_Fraude": v_ideal["tp"]["pct"], "TN_Pct_Real_Nao_Fraude": v_ideal["tn"]["pct"], "FP_Pct_Real_Nao_Fraude": v_ideal["fp"]["pct"]},
    ])

    tm_html = tabela_metricas.copy()
    for c in tm_html.columns:
        if pd.api.types.is_float_dtype(tm_html[c]):
            tm_html[c] = tm_html[c].apply(lambda z: formatar_float_html(z, 6))

    html_metricas = secao_tabela(f"Métricas do Rank {rank}", gerar_tabela_html(tm_html, f"tabela_metricas_rank_{rank}"), f"tabela_metricas_rank_{rank}", f"{prefixo}_metricas.csv")

    html_rank = f'''
    <section class="rank-section" id="rank-{rank}">
      <h1>Rank {rank} - t-SNE 1D | Perplexity = {perplexity:g}</h1>
      <div class="info-box"><div class="info-grid">
        <div class="info-item"><div class="info-label">Rank</div><div class="info-value">{rank}</div></div>
        <div class="info-item"><div class="info-label">Feature visualizada</div><div class="info-value">TSNE_1</div></div>
        <div class="info-item"><div class="info-label">Perplexity</div><div class="info-value">{perplexity:g}</div></div>
        <div class="info-item"><div class="info-label">Max Iter</div><div class="info-value">{max_iter}</div></div>
        <div class="info-item"><div class="info-label">Init</div><div class="info-value">{html.escape(str(init))}</div></div>
        <div class="info-item"><div class="info-label">Random State t-SNE</div><div class="info-value">{random_state_tsne}</div></div>
        <div class="info-item"><div class="info-label">Melhor Ponto de Corte</div><div class="info-value">{melhor_corte:.6f}</div></div>
        <div class="info-item"><div class="info-label">Ponto de Corte Médio</div><div class="info-value">{corte_medio:.6f}</div></div>
        <div class="info-item"><div class="info-label">AUC-PR</div><div class="info-value">{float(valor_linha(linha, 'AUC_PR', auc_pr_calc)):.6f}</div></div>
        <div class="info-item"><div class="info-label">MCC</div><div class="info-value">{float(valor_linha(linha, 'MCC', mcc_calc)):.6f}</div></div>
        <div class="info-item"><div class="info-label">Score Final</div><div class="info-value">{float(valor_linha(linha, 'Score_Final', np.nan)):.6f}</div></div>
        <div class="info-item"><div class="info-label">Tempo de recriação</div><div class="info-value">{tempo}</div></div>
      </div></div>
      {html_metricas}
      {html_matriz(f'Matriz de Confusão (%) - t-SNE 1D - Perplexity {perplexity:g} - Melhor Ponto de Corte', v_melhor, False, imgs['matriz_melhor_corte'], f'{prefixo}_matriz_melhor_corte.png')}
      {html_matriz(f'Matriz de Confusão (%) - t-SNE 1D - Perplexity {perplexity:g} - Ponto de Corte 0.5', v_medio, False, imgs['matriz_corte_medio'], f'{prefixo}_matriz_corte_medio.png')}
      {html_matriz('Matriz de Confusão Ideal (%)', v_ideal, True, imgs['matriz_ideal'], f'{prefixo}_matriz_ideal.png')}
      {secao_imagem(f'Boxplot por Classe - {titulo_base}', imgs['boxplot'], f'{prefixo}_boxplot.png', 'Boxplot')}
      {secao_imagem(f'Representação 1D com 100% dos Dados - {titulo_base}', imgs['representacao_1d'], f'{prefixo}_representacao_1d.png', 'Representação 1D')}
      {secao_imagem(f'Correlação de Spearman entre TSNE_1 e Target - Perplexity = {perplexity:g}', imgs['spearman'], f'{prefixo}_spearman.png', 'Spearman')}
      {secao_imagem(f'Responsabilidades Estimadas pela GMM - Melhor Ponto de Corte - {titulo_base}', imgs['responsabilidades_melhor_corte'], f'{prefixo}_responsabilidades_melhor_corte.png', 'Responsabilidades melhor corte')}
      {secao_imagem(f'Responsabilidades Estimadas pela GMM - Ponto de Corte Médio - {titulo_base}', imgs['responsabilidades_corte_medio'], f'{prefixo}_responsabilidades_corte_medio.png', 'Responsabilidades corte médio')}
    </section>
    '''

    print(f"Rank {rank} finalizado em {tempo}")
    return {"rank": rank, "perplexity": perplexity, "html": html_rank, "tabela_metricas": tabela_metricas, "tabela_matrizes": tabela_matrizes}

def gerar_relatorio_1x1_tsne_unificado(
    ranks=(1, 2, 3),
    arquivo_dados="creditcard.csv",
    arquivo_scores="1x1_tsne_score.csv",
    pasta_saida=".",
    nome_arquivo_html="1x1_tsne_visu_scores.html",
    pasta_latex="1x1_tsne_visu_scores",
    exportar_latex=True,
    target_name=None,
    features_tsne=None,
    verbose_tsne=0,
):
    t0 = time.time()
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)
    caminho_html = pasta_saida / nome_arquivo_html
    caminho_latex = pasta_saida / pasta_latex
    caminho_latex.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(arquivo_dados)
    scores = pd.read_csv(arquivo_scores)

    if target_name is None:
        target_name = detectar_target(df)
    if target_name not in df.columns:
        raise ValueError(f"Target '{target_name}' não encontrado.")

    if features_tsne is None:
        features_tsne = [c for c in df.columns if c != target_name and pd.api.types.is_numeric_dtype(df[c])]
    if not features_tsne:
        raise ValueError("Nenhuma feature numérica encontrada para recriar o t-SNE.")

    if "Posicao_Rank" not in scores.columns:
        if "Score_Final" not in scores.columns:
            raise ValueError("O arquivo de scores precisa ter 'Posicao_Rank' ou 'Score_Final'.")
        scores = scores.sort_values("Score_Final", ascending=False).reset_index(drop=True)
        scores["Posicao_Rank"] = np.arange(1, len(scores) + 1)

    print("=" * 80)
    print("RELATÓRIO VISUAL t-SNE 1D - TOP RANKS")
    print("=" * 80)
    print(f"Arquivo de dados: {arquivo_dados}")
    print(f"Arquivo de scores: {arquivo_scores}")
    print(f"HTML final: {nome_arquivo_html}")
    print(f"Pasta LaTeX/imagens: {pasta_latex}")
    print(f"Target: {target_name}")
    print(f"Features usadas para recriar o t-SNE: {len(features_tsne)}")
    print(f"Ranks solicitados: {ranks}")
    print("=" * 80)

    resultados = []
    for r in ranks:
        resultados.append(processar_rank(r, df, scores, target_name, features_tsne, caminho_latex, verbose_tsne))

    tabela_metricas = pd.concat([r["tabela_metricas"] for r in resultados], ignore_index=True)
    tabela_matrizes = pd.concat([r["tabela_matrizes"] for r in resultados], ignore_index=True)

    if exportar_latex:
        exportar_latex_fn = globals()["exportar_latex"]
        exportar_latex_fn(caminho_latex, tabela_metricas, tabela_matrizes)

    tm_html = tabela_metricas.copy()
    mt_html = tabela_matrizes.copy()
    for dff in [tm_html, mt_html]:
        for c in dff.columns:
            if pd.api.types.is_float_dtype(dff[c]):
                dff[c] = dff[c].apply(lambda z: formatar_float_html(z, 6))

    html_resumo_metricas = secao_tabela("Tabela Geral de Métricas dos Ranks", gerar_tabela_html(tm_html, "tabela_metricas_ranks"), "tabela_metricas_ranks", "tabela_metricas_ranks.csv")
    html_resumo_matrizes = secao_tabela("Tabela Geral das Matrizes de Confusão dos Ranks", gerar_tabela_html(mt_html, "tabela_matrizes_confusao_ranks"), "tabela_matrizes_confusao_ranks", "tabela_matrizes_confusao_ranks.csv")

    navegacao = "\n".join([f'<a href="#rank-{r["rank"]}">Rank {r["rank"]} - t-SNE 1D | Perplexity {r["perplexity"]:g}</a>' for r in resultados])
    html_ranks = "\n".join([r["html"] for r in resultados])

    html_final = f'''
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
      <meta charset="UTF-8">
      <title>Relatório t-SNE 1D - Top Ranks</title>
      <style>
        body {{ font-family: Arial, Helvetica, sans-serif; background: #f4f6f8; color: #020617; margin: 0; padding: 32px; }}
        .container {{ max-width: 1250px; margin: 0 auto; }}
        h1 {{ text-align: center; margin-bottom: 28px; color: #020617; }}
        .main-title {{ font-size: 34px; margin-bottom: 14px; }}
        .nav-box {{ background: #fff; border-radius: 16px; padding: 18px; margin-bottom: 28px; box-shadow: 0 8px 24px rgba(15,23,42,.08); display: flex; justify-content: center; gap: 12px; flex-wrap: wrap; }}
        .nav-box a {{ text-decoration: none; background: #eff6ff; color: #1e3a8a; border: 1px solid #bfdbfe; padding: 8px 12px; border-radius: 999px; font-weight: 800; font-size: 13px; }}
        .rank-section {{ margin-top: 46px; padding-top: 12px; border-top: 4px solid #cbd5e1; }}
        .info-box, .matrix-card, .plot-card {{ background: #fff; border-radius: 16px; padding: 24px; margin-bottom: 32px; box-shadow: 0 8px 24px rgba(15,23,42,.08); }}
        .info-grid {{ display: grid; grid-template-columns: repeat(3, 1fr); gap: 14px; margin-top: 14px; }}
        .info-item {{ background: #f8fafc; border: 1px solid #e2e8f0; border-radius: 12px; padding: 12px 14px; }}
        .info-label {{ font-size: 13px; font-weight: 700; color: #475569; margin-bottom: 6px; }}
        .info-value {{ font-size: 18px; font-weight: 800; color: #020617; font-family: Consolas, Monaco, monospace; }}
        .matrix-card h2, .plot-card h2 {{ text-align: center; margin-top: 0; margin-bottom: 24px; color: #020617; font-size: 22px; }}
        .section-header {{ display: flex; align-items: center; justify-content: center; gap: 14px; flex-wrap: wrap; margin-bottom: 18px; }}
        .section-header h2 {{ margin: 0; }}
        .section-actions-only {{ display: flex; justify-content: flex-end; margin-bottom: 12px; }}
        .download-btn {{ border: 1px solid #bfdbfe; background: #eff6ff; color: #1e3a8a; padding: 8px 12px; border-radius: 10px; font-weight: 800; font-size: 13px; cursor: pointer; text-decoration: none; display: inline-block; }}
        .download-btn:hover {{ background: #dbeafe; }}
        .table-wrapper {{ overflow-x: auto; border: 1px solid #e2e8f0; border-radius: 12px; max-height: 580px; overflow-y: auto; }}
        .data-table {{ width: 100%; border-collapse: collapse; font-size: 13px; margin-top: 0; }}
        .data-table th {{ background: #0f172a; color: white; padding: 10px 8px; text-align: left; position: sticky; top: 0; z-index: 1; }}
        .data-table td {{ border-bottom: 1px solid #e2e8f0; padding: 8px; color: #020617; white-space: nowrap; }}
        .data-table tr:nth-child(even) {{ background: #f8fafc; }}
        .matrix-area {{ display: flex; align-items: center; justify-content: center; gap: 34px; }}
        .matrix-wrapper {{ display: grid; grid-template-columns: 180px 1fr 1fr; grid-template-rows: 48px 190px 190px; width: 950px; }}
        .corner {{ background: transparent; }}
        .x-label {{ display: flex; align-items: center; justify-content: center; font-size: 19px; font-weight: 700; color: #020617; border-bottom: 1px solid #e5e7eb; }}
        .y-label {{ display: flex; align-items: center; justify-content: flex-end; padding-right: 18px; font-size: 19px; font-weight: 700; color: #020617; }}
        .cell {{ display: flex; flex-direction: column; align-items: center; justify-content: center; min-height: 180px; border: 1px solid #e5e7eb; font-size: 20px; text-align: center; color: #020617 !important; }}
        .pct {{ font-size: 30px; font-weight: 900; margin-bottom: 4px; color: #020617 !important; }}
        .count {{ font-size: 24px; font-weight: 900; margin-bottom: 8px; color: #020617 !important; }}
        .cell-desc {{ font-size: 13px; font-weight: 700; color: #020617 !important; }}
        .q95 {{ background: #08306b; }} .q85 {{ background: #08519c; }} .q70 {{ background: #2171b5; }} .q50 {{ background: #6baed6; }} .q30 {{ background: #c6dbef; }} .q10 {{ background: #eff6ff; }}
        .legend {{ position: relative; display: flex; flex-direction: column; align-items: center; min-width: 115px; }}
        .legend-title {{ font-weight: 800; font-size: 15px; margin-bottom: 10px; color: #020617; }}
        .colorbar {{ width: 30px; height: 310px; border-radius: 16px; background: linear-gradient(to bottom,#08306b 0%,#08519c 18%,#2171b5 36%,#6baed6 58%,#c6dbef 78%,#eff6ff 100%); border: 1px solid #cbd5e1; }}
        .legend-label-top {{ position: absolute; top: 43px; left: 78px; font-size: 13px; font-weight: 800; color: #020617; }}
        .legend-label-bottom {{ position: absolute; top: 335px; left: 78px; font-size: 13px; font-weight: 800; color: #020617; }}
        .plot-img {{ display: block; max-width: 100%; margin: 0 auto; border-radius: 12px; border: 1px solid #e2e8f0; }}
        @media (max-width: 1100px) {{ .info-grid {{ grid-template-columns: repeat(2,1fr); }} .matrix-area {{ flex-direction: column; }} .matrix-wrapper {{ width: 100%; grid-template-columns: 150px 1fr 1fr; }} }}
        @media (max-width: 700px) {{ body {{ padding: 16px; }} .info-grid {{ grid-template-columns: 1fr; }} .matrix-wrapper {{ grid-template-columns: 120px 1fr 1fr; grid-template-rows: 48px 160px 160px; }} .pct {{ font-size: 22px; }} .count {{ font-size: 18px; }} .y-label, .x-label {{ font-size: 14px; }} }}
      </style>
    </head>
    <body>
      <div class="container">
        <h1 class="main-title">Relatório Unificado t-SNE 1D - Top Ranks</h1>
        <div class="nav-box">{navegacao}</div>
        {html_ranks}
        {html_resumo_metricas}
        {html_resumo_matrizes}
      </div>
      <script>
        function limparTextoCSV(texto) {{
          if (texto === null || texto === undefined) return "";
          texto = String(texto).replace(/\n/g, " ").replace(/\s+/g, " ").trim();
          if (texto.includes(";") || texto.includes('"')) texto = '"' + texto.replace(/"/g, '""') + '"';
          return texto;
        }}
        function baixarTabelaCSV(tableId, filename) {{
          const tabela = document.getElementById(tableId);
          if (!tabela) {{ alert("Tabela não encontrada: " + tableId); return; }}
          const linhas = [];
          tabela.querySelectorAll("tr").forEach(function(row) {{
            const celulas = Array.from(row.querySelectorAll("th, td"));
            linhas.push(celulas.map(celula => limparTextoCSV(celula.innerText)).join(";"));
          }});
          const csv = "\ufeff" + linhas.join("\n");
          const blob = new Blob([csv], {{ type: "text/csv;charset=utf-8;" }});
          const url = URL.createObjectURL(blob);
          const link = document.createElement("a");
          link.href = url; link.download = filename; document.body.appendChild(link); link.click(); document.body.removeChild(link);
          URL.revokeObjectURL(url);
        }}
      </script>
    </body>
    </html>
    '''

    caminho_html.write_text(html_final, encoding="utf-8")
    print("=" * 80)
    print("RELATÓRIO t-SNE 1D UNIFICADO GERADO COM SUCESSO")
    print("=" * 80)
    print(f"HTML salvo em: {caminho_html.resolve()}")
    print(f"Arquivos LaTeX/imagens salvos em: {caminho_latex.resolve()}")
    print(f"Tempo total: {formatar_tempo(time.time() - t0)}")
    print("=" * 80)
    return {"caminho_html": caminho_html, "caminho_latex": caminho_latex, "tabela_metricas": tabela_metricas, "tabela_matrizes": tabela_matrizes}


resultado_1x1_tsne_visu_scores = gerar_relatorio_1x1_tsne_unificado(
    ranks=(1, 2, 3),
    arquivo_dados="creditcard.csv",
    arquivo_scores="1x1_tsne_score.csv",
    pasta_saida=".",
    nome_arquivo_html="1x1_tsne_visu_scores.html",
    pasta_latex="1x1_tsne_visu_scores",
    exportar_latex=True,
    target_name=None,
    features_tsne=None,
    verbose_tsne=0,
)

resultado_1x1_tsne_visu_scores["caminho_html"]
